<p align="center"><a href="https://www.plus2net.com/python/pandas-dataframe-plot-mysql.php"><img src="https://www.plus2net.com/images/top2.jpg" width="300" alt="plus2net Python tutorials"></a></p>

# Plotting Graphs from MySQL Database using Pandas DataFrame

**Tutorial & Full Documentation:** [https://www.plus2net.com/python/pandas-dataframe-plot-mysql.php](https://www.plus2net.com/python/pandas-dataframe-plot-mysql.php)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This interactive notebook accompanies the plus2net tutorial on plotting from SQL databases. Execute each cell to extract aggregated database records with pd.read_sql() and generate visualizations.



### 1. Database Setup and Query Execution
Create a simulated SQLite database table and load student records with `pd.read_sql()`.


In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

conn = sqlite3.connect(":memory:")
students = pd.DataFrame({
    "id": range(1, 11),
    "name": ["John", "Ravi", "Sara", "Alex", "Nina", "Mike", "Tara", "Leo", "Maya", "Omar"],
    "class1": ["Four", "Three", "Four", "Five", "Three", "Four", "Five", "Three", "Four", "Five"],
    "mark": [75, 82, 90, 65, 88, 70, 95, 60, 85, 78],
    "gender": ["male", "male", "female", "male", "female", "male", "female", "male", "female", "male"]
})
students.to_sql("student", conn, index=False)
print("Student table populated!")


### 2. Pie Chart from SQL Group By Aggregation
Query class distribution and plot a pie chart.


In [ ]:
q_pie = "SELECT class1, COUNT(*) as number FROM student GROUP BY class1"
df_pie = pd.read_sql(q_pie, conn)
print(df_pie)

df_pie.plot.pie(
    y="number",
    labels=df_pie["class1"].tolist(),
    autopct="%1.0f%%",
    figsize=(5, 5),
    title="Students by Class"
)
plt.ylabel("")
plt.show()


### 3. Multi-Metric Bar Chart and Stacked Bar with CASE
Compare average and maximum scores, then plot gender composition using SQL CASE.


In [ ]:
q_bar = "SELECT class1, AVG(mark) as avg_mark, MAX(mark) as max_mark FROM student GROUP BY class1"
df_bar = pd.read_sql(q_bar, conn)

df_bar.plot.bar(x="class1", y=["avg_mark", "max_mark"], figsize=(6.5, 4), title="Average and Max Marks")
plt.ylabel("Mark")
plt.show()

q_stack = """
SELECT class1,
SUM(CASE WHEN gender='male' THEN 1 ELSE 0 END) as Male,
SUM(CASE WHEN gender='female' THEN 1 ELSE 0 END) as Female
FROM student GROUP BY class1
"""
df_stack = pd.read_sql(q_stack, conn)
df_stack.plot.bar(x="class1", stacked=True, figsize=(6, 4), title="Gender Distribution by Class")
plt.ylabel("Student Count")
plt.show()


### 4. Practical Workflow: Retail Orders Dashboard
Generate a two-panel dashboard from SQL aggregated orders.


In [ ]:
orders = pd.DataFrame({
    "order_id": range(1, 9),
    "category": ["Electronics", "Apparel", "Electronics", "Home", "Apparel", "Home", "Electronics", "Home"],
    "amount": [240, 85, 410, 130, 95, 180, 320, 210]
})
orders.to_sql("orders", conn, index=False)

q_orders = """
SELECT category,
COUNT(*) as order_count,
SUM(amount) as total_revenue,
AVG(amount) as avg_order_val
FROM orders
GROUP BY category
ORDER BY total_revenue DESC
"""
df_report = pd.read_sql(q_orders, conn)
print(df_report)

assert len(df_report) == 3
assert df_report["total_revenue"].sum() == 1670

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
df_report.plot.bar(x="category", y="total_revenue", ax=axes[0], title="Total Revenue ($)")
df_report.plot.pie(y="order_count", labels=df_report["category"], ax=axes[1], autopct="%1.0f%%", title="Order Share")
axes[1].set_ylabel("")
plt.tight_layout()
plt.show()


### 5. Hands-on Exercise: Customer Balance by Tier
Aggregate customer account balances and plot a horizontal bar chart.


In [ ]:
cust_df = pd.DataFrame({
    "id": range(1, 6),
    "tier": ["Gold", "Silver", "Gold", "Bronze", "Silver"],
    "balance": [6000, 3500, 4500, 1000, 1500]
})
cust_df.to_sql("customers", conn, index=False)

q_cust = """
SELECT tier,
COUNT(*) as customer_count,
SUM(balance) as total_balance
FROM customers
GROUP BY tier
ORDER BY total_balance DESC
"""
result_df = pd.read_sql(q_cust, conn)
print(result_df)

assert result_df["total_balance"].sum() == 16500

result_df.plot.barh(
    x="tier",
    y="total_balance",
    grid=True,
    figsize=(6.5, 3.5),
    title="Account Balance by Customer Tier"
)
plt.xlabel("Total Balance ($)")
plt.show()
